<a href="https://colab.research.google.com/github/shini-080507/insurance-/blob/main/Insurance_Policy_%26_Billing_Knowledge_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib openai pypdf

import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from pypdf import PdfReader
from openai import OpenAI

data = {
    "age": [25, 45, 32, 56, 40, 29, 61, 35, 48, 52,
            27, 43, 67, 38, 50, 31, 59, 42, 36, 64],

    "hospitalization_days": [2, 7, 3, 12, 8, 1, 10, 4, 6, 9,
                             2, 5, 15, 3, 7, 2, 11, 6, 4, 13],

    "medical_expenses": [15000, 85000, 25000, 150000, 95000,
                         10000, 180000, 35000, 70000, 110000,
                         18000, 60000, 220000, 30000, 90000,
                         20000, 160000, 75000, 45000, 200000],

    "previous_claims": [0, 2, 1, 3, 2, 0, 4, 1, 2, 3,
                        0, 1, 5, 1, 2, 0, 3, 2, 1, 4],

    "severity": [
        "Low", "High", "Low", "High", "Medium",
        "Low", "High", "Low", "Medium", "High",
        "Low", "Medium", "High", "Low", "Medium",
        "Low", "High", "Medium", "Low", "High"
    ]
}

df = pd.DataFrame(data)

df.head()

mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

reverse_mapping = {
    0: "Low",
    1: "Medium",
    2: "High"
}

features = [
    "age",
    "hospitalization_days",
    "medical_expenses",
    "previous_claims"
]

X = df[features]
y = df["severity"].map(mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

prediction = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    prediction
)

print("Accuracy:", round(accuracy * 100, 2), "%")

new_claim = pd.DataFrame({
    "age": [45],
    "hospitalization_days": [7],
    "medical_expenses": [85000],
    "previous_claims": [2]
})

result = model.predict(new_claim)[0]

print("Predicted Claim Severity:",
      reverse_mapping[result])

policy = {
    "room_charge_limit": 5000,
    "surgery_covered": True,
    "doctor_charges_covered": True,
    "medicine_covered": True,
    "diagnostics_covered": True,
    "non_medical_expenses": False,
    "maximum_coverage": 100000
}

print("Policy rules loaded successfully.")

def extract_bill_text(filename):

    reader = PdfReader(filename)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


print("""
Medical Bill Parsing Workflow

Medical Bill
     ↓
PDF Text Extraction
     ↓
LLM Document Parser
     ↓
Structured JSON
     ↓
Policy Rule Verification
""")

from getpass import getpass

api_key = getpass("Enter OpenAI API Key: ")

client = OpenAI(api_key=api_key)


def parse_medical_bill(bill_text):

    prompt = f"""
Extract the following information from this medical bill.

Return ONLY valid JSON.

Fields:
patient_name
hospital_name
policy_number
treatment
hospitalization_days
room_charges
surgery_charges
doctor_charges
medicine_charges
diagnostic_charges
non_medical_expenses
total_bill_amount

Medical Bill:
{bill_text}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return json.loads(
        response.choices[0].message.content
    )

print("""
================================================
MEDICAL BILL / POLICY PARSING WORKFLOW
================================================

1. Medical bill is received
              ↓
2. Text is extracted from the document
              ↓
3. LLM identifies important bill fields
              ↓
4. Information is converted into JSON
              ↓
5. Extracted expenses are compared
   with insurance policy rules
              ↓
6. Covered and non-covered expenses
   are identified
              ↓
7. Claim severity is predicted using
   the Random Forest model
              ↓
8. Final claim analysis is generated

================================================
SYSTEM OUTPUT
================================================

Claim Severity:
Low / Medium / High

Medical Bill:
Structured JSON

Policy Verification:
Covered / Non-Covered

================================================
""")

policy_kb = {
    "room_charges": {
        "covered": True,
        "limit": 5000
    },
    "surgery": {
        "covered": True
    },
    "doctor_charges": {
        "covered": True
    },
    "medicines": {
        "covered": True
    },
    "diagnostics": {
        "covered": True
    },
    "non_medical_expenses": {
        "covered": False
    },
    "maximum_coverage": 100000
}

print("Insurance Policy Knowledge Base")
print(json.dumps(policy_kb, indent=2))

billing_kb = {
    "patient_age": None,
    "hospitalization_days": None,
    "room_charges": 0,
    "surgery_charges": 0,
    "doctor_charges": 0,
    "medicine_charges": 0,
    "diagnostic_charges": 0,
    "non_medical_expenses": 0,
    "total_amount": 0
}

print("Medical Billing Knowledge Base")
print(json.dumps(billing_kb, indent=2))

def verify_bill_with_policy(bill):
    result = {}

    result["room_charges"] = (
        bill.get("room_charges", 0) <= policy_kb["room_charges"]["limit"]
    )

    result["surgery"] = policy_kb["surgery"]["covered"]
    result["doctor_charges"] = policy_kb["doctor_charges"]["covered"]
    result["medicines"] = policy_kb["medicines"]["covered"]
    result["diagnostics"] = policy_kb["diagnostics"]["covered"]

    result["non_medical_expenses"] = (
        policy_kb["non_medical_expenses"]["covered"]
    )

    result["within_maximum_coverage"] = (
        bill.get("total_amount", 0) <= policy_kb["maximum_coverage"]
    )

    return result

print("Policy and Billing Knowledge Base Connected")

document_prompt = """
Extract the following information from the medical bill.

Return ONLY valid JSON with these fields:
{
  "patient_age": 0,
  "hospitalization_days": 0,
  "room_charges": 0,
  "surgery_charges": 0,
  "doctor_charges": 0,
  "medicine_charges": 0,
  "diagnostic_charges": 0,
  "non_medical_expenses": 0,
  "total_amount": 0
}

If any value is missing, use 0.
Do not guess values that are not present in the document.
"""

claim_explanation_prompt = """
Explain the insurance claim analysis in simple language.

Include:
1. Claim severity
2. Total bill amount
3. Covered expenses
4. Non-covered expenses
5. Policy limit status
6. Reason for the predicted severity

Do not invent information.
Base the explanation only on the extracted bill,
insurance policy rules, and model prediction.
"""